In [2]:
import os
import glob
import pandas as pd
import numpy as np

# Pfad zum Hauptordner (ggf. anpassen, falls der Ordner woanders liegt)
BASE_DIR = "./test_data"

In [6]:
def load_recording_csvs(folder_path):
    """Sucht und lädt alle CSV-Dateien mit 'data_recording' im Namen."""
    search_path = os.path.join(folder_path, "*data_recording*.csv")
    files = glob.glob(search_path)
    dfs = {}
    
    for f in files:
        filename = os.path.basename(f)
        df = pd.read_csv(f)
        # Bereinige Whitespaces in den Spaltennamen und Schritten
        df.columns = df.columns.str.strip()
        if 'step' in df.columns:
            df['step'] = df['step'].str.strip()
        dfs[filename] = df
    return dfs

def calculate_cross_file_error(dfs_dict):
    """Berechnet Mittelwert und Standardabweichung (Fehler) zwischen allen Dokumenten."""
    if len(dfs_dict) < 2:
        print("-> Info: Weniger als 2 Dateien im Ordner. Kein Dokumenten-Vergleich möglich.")
        return None
    
    # Spalten A_1 bis A_7 dynamisch filtern
    val_cols = [c for c in list(dfs_dict.values())[0].columns if c.startswith('A_')]
    
    # Alle DataFrames kombinieren und über den Zeilenindex gruppieren
    combined = pd.concat(dfs_dict.values(), keys=dfs_dict.keys(), names=['Datei', 'Zeile'])
    
    # Statistiken berechnen
    mean_df = combined.groupby(level='Zeile')[val_cols].mean().add_suffix('_mean')
    std_df = combined.groupby(level='Zeile')[val_cols].std().add_suffix('_std_error')
    
    # Ergebnisse zusammenführen
    result = pd.concat([std_df], axis=1) #mean_df,
    
    # 'step'-Spalte aus der ersten Datei zur Orientierung wieder hinzufügen
    first_df = list(dfs_dict.values())[0]
    if 'step' in first_df.columns:
        result.insert(0, 'step', first_df['step'])
        
    return result

In [7]:
print("=== AUSWERTUNG: STANDARD-ORDNER (OHNE LOOP) ===")

for folder in os.listdir(BASE_DIR):
    folder_path = os.path.join(BASE_DIR, folder)
    
    # Nur Ordner verarbeiten, die NICHT 'loop' im Namen haben
    if os.path.isdir(folder_path) and "loop" not in folder.lower():
        print(f"\n📂 Verarbeite Ordner: {folder}")
        dfs = load_recording_csvs(folder_path)
        
        if not dfs:
            print("   Keine 'data_recording' CSV-Dateien gefunden.")
            continue
            
        print(f"   {len(dfs)} Datei(en) gefunden. Berechne Fehler zwischen den Dokumenten...")
        error_df = calculate_cross_file_error(dfs)
        
        if error_df is not None:
            # Zeigt die ersten Zeilen des Fehler-Berichts im Notebook an
            display(error_df.head(10))
            # Optional: Als CSV speichern mit: error_df.to_csv(f"{folder}_vergleich.csv")

=== AUSWERTUNG: STANDARD-ORDNER (OHNE LOOP) ===

📂 Verarbeite Ordner: dreieck_default
   3 Datei(en) gefunden. Berechne Fehler zwischen den Dokumenten...


,step,A_1_std_error,A_2_std_error,A_3_std_error,A_4_std_error,A_5_std_error,A_6_std_error,A_7_std_error
Zeile,,,,,,,,
0,data_recording_center,0.030688,0.058478,0.154715,0.002616,0.059256,0.055508,0.005020
1,data_recording_front,0.026022,0.018314,0.124168,0.000212,0.121905,0.007283,0.004808
2,data_recording_left,0.100056,0.017395,0.095672,0.012657,0.033305,0.023547,0.009334
3,data_recording_right,0.004667,0.124097,0.102036,0.057417,0.056427,0.009122,0.000707
4,data_recording_back,0.028638,0.006152,0.048013,0.026375,0.015556,0.006081,0.000566



📂 Verarbeite Ordner: dreieck_sleep
   3 Datei(en) gefunden. Berechne Fehler zwischen den Dokumenten...


,step,A_1_std_error,A_2_std_error,A_3_std_error,A_4_std_error,A_5_std_error,A_6_std_error,A_7_std_error
Zeile,,,,,,,,
0,data_recording_center,0.029639,0.087499,0.122454,0.042205,0.237461,0.011212,0.004210
1,data_recording_front,0.456194,0.086567,0.164401,0.022418,0.287278,0.012345,0.027403
2,data_recording_left,0.060593,0.075672,0.048830,0.049764,0.017432,0.045666,0.007130
3,data_recording_right,0.272760,0.146654,0.051829,0.119424,0.029405,0.087921,0.015952
4,data_recording_back,0.108717,0.058668,0.069786,0.061335,0.019805,0.024967,0.002914


In [13]:
print("=== AUSWERTUNG: LOOP-ORDNER (MIT WIEDERHOLUNGEN) ===")

for folder in os.listdir(BASE_DIR):
    folder_path = os.path.join(BASE_DIR, folder)
    
    # Nur Ordner verarbeiten, die 'loop' im Namen haben
    if os.path.isdir(folder_path) and "loop" in folder.lower():
        print(f"\n🔄 📂 Verarbeite LOOP-Ordner: {folder}")
        print("-" * 50)
        
        dfs = load_recording_csvs(folder_path)
        
        if not dfs:
            print("   Keine 'data_recording' CSV-Dateien gefunden.")
            continue
            
        # Schritt 1: Vergleich ZWISCHEN den Dokumenten
        print("\n1️⃣ Fehlervergleich ZWISCHEN den Dokumenten im Loop-Ordner:")
        error_df = calculate_cross_file_error(dfs)
        if error_df is not None:
            display(error_df)
            
        # Schritt 2: Analyse der 5 Punkte INNERHALB jedes einzelnen Dokuments
        print("\n2️⃣ Analyse der wiederholten Punkte INNERHALB der einzelnen Dokumente:")
        for filename, df in dfs.items():
            print(f"   📄 Datei: {filename}")
            val_cols = [c for c in df.columns if c.startswith('A_')]
            
            if 'step' in df.columns:
                # Gruppiert nach den 5 angefahrenen Punkten (z.B. data_recording_center, etc.)
                # und berechnet Mittelwert sowie Standardabweichung der Wiederholungen
                loop_analysis = df.groupby('step')[val_cols].agg([ 'std']) #'mean',
                
                # Spaltennamen für die Übersichtlichkeit flachklopfen
                loop_analysis.columns = [f"{col}_{stat}" for col, stat in loop_analysis.columns]
                display(loop_analysis)
            else:
                print("   ⚠️ Warnung: Keine 'step'-Spalte für die Loop-Auswertung gefunden!")

=== AUSWERTUNG: LOOP-ORDNER (MIT WIEDERHOLUNGEN) ===

🔄 📂 Verarbeite LOOP-Ordner: dreieick_loop
--------------------------------------------------

1️⃣ Fehlervergleich ZWISCHEN den Dokumenten im Loop-Ordner:


,step,A_1_std_error,A_2_std_error,A_3_std_error,A_4_std_error,A_5_std_error,A_6_std_error,A_7_std_error
Zeile,,,,,,,,
0,data_recording_center,0.058973,0.003889,0.284823,0.014991,0.122895,0.022698,0.009192
1,data_recording_front,0.431547,0.141421,0.416557,0.116390,0.088035,0.022627,0.005091
2,data_recording_left,0.013294,0.085065,0.005586,0.027224,0.051760,0.025880,0.000283
3,data_recording_right,0.119430,0.364443,0.073327,0.191272,0.093974,0.000283,0.001838
4,data_recording_back,0.035567,0.031608,0.023122,0.028567,0.004879,0.017819,0.000707
5,data_recording_center,0.111369,0.072620,0.001697,0.011950,0.008061,0.347967,0.010889
6,data_recording_front,0.517249,0.188090,0.031891,0.078277,0.295288,0.006718,0.047871
7,data_recording_left,0.090722,0.053882,0.027224,0.109814,0.041507,0.045679,0.000141
8,data_recording_right,0.411395,0.091429,0.259296,0.062438,0.168504,0.007142,0.008980



2️⃣ Analyse der wiederholten Punkte INNERHALB der einzelnen Dokumente:
   📄 Datei: data_recording_20260609_161846_1.csv


,A_1_std,A_2_std,A_3_std,A_4_std,A_5_std,A_6_std,A_7_std
step,,,,,,,
data_recording_back,0.100540,0.058750,0.098080,0.046173,0.020821,0.010346,0.002937
data_recording_center,0.074099,0.046699,0.356628,0.046249,0.115875,0.458288,0.023315
data_recording_front,0.353263,0.148022,0.136468,0.160220,0.233291,0.053498,0.032979
data_recording_left,0.086492,0.093425,0.051982,0.014508,0.030138,0.018592,0.001861
data_recording_right,0.096997,0.187947,0.104213,0.060781,0.155167,0.040677,0.017773


   📄 Datei: data_recording_20260609_162253_2.csv


...

,step,A_1_std_error,A_2_std_error,A_3_std_error,A_4_std_error,A_5_std_error,A_6_std_error,A_7_std_error
Zeile,,,,,,,,
0,data_recording_center,0.029840,0.076014,0.060599,0.023264,0.324421,0.006930,0.006364
1,data_recording_front,0.015274,0.029204,0.152382,0.207748,0.038325,0.097793,0.015556
2,data_recording_left,0.144815,0.049285,0.170978,0.043487,0.057912,0.045255,0.000424
3,data_recording_right,0.244659,0.002899,0.108753,0.023900,0.022698,0.054659,0.005798
4,data_recording_back,0.082237,0.055649,0.055154,0.045891,0.037547,0.009829,0.002687
5,data_recording_center,0.436992,0.052467,0.073751,0.136684,0.036062,0.117875,0.035780
6,data_recording_front,0.570989,0.013223,0.071559,0.094116,0.355887,0.013647,0.034012
7,data_recording_left,0.045326,0.040588,0.002970,0.051407,0.027648,0.012304,0.001131
8,data_recording_right,0.458771,0.072478,0.180029,0.047235,0.011172,0.003536,0.000636



2️⃣ Analyse der wiederholten Punkte INNERHALB der einzelnen Dokumente:
   📄 Datei: data_recording_20260609_160943_1.csv


,A_1_std,A_2_std,A_3_std,A_4_std,A_5_std,A_6_std,A_7_std
step,,,,,,,
data_recording_back,0.048727,0.082443,0.064782,0.074108,0.024645,0.037265,0.001656
data_recording_center,0.057639,0.045933,0.209605,0.016563,0.346867,0.338020,0.037254
data_recording_front,0.401551,0.158690,0.044255,0.196511,0.259788,0.081358,0.038400
data_recording_left,0.011066,0.063579,0.041863,0.036960,0.015788,0.056169,0.001609
data_recording_right,0.164504,0.017180,0.046345,0.070076,0.014351,0.061354,0.002307


   📄 Datei: data_recording_20260609_161314_2.csv


,A_1_std,A_2_std,A_3_std,A_4_std,A_5_std,A_6_std,A_7_std
step,,,,,,,
data_recording_back,0.105022,0.044505,0.065438,0.069355,0.028634,0.008792,0.002627
data_recording_center,0.298984,0.081247,0.363597,0.140528,0.122635,0.432446,0.032692
data_recording_front,0.076153,0.097638,0.015387,0.037376,0.050120,0.019347,0.002108
data_recording_left,0.128355,0.085903,0.085100,0.031304,0.016489,0.014789,0.002122
data_recording_right,0.374377,0.067326,0.196639,0.016211,0.073494,0.026446,0.023437
